### RAG 

In [1]:
from dotenv import load_dotenv
load_dotenv

<function dotenv.main.load_dotenv(dotenv_path: Union[str, ForwardRef('os.PathLike[str]'), NoneType] = None, stream: Optional[IO[str]] = None, verbose: bool = False, override: bool = False, interpolate: bool = True, encoding: Optional[str] = 'utf-8') -> bool>

In [127]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate

loader = PyPDFLoader("../Data/medical_report.pdf" )
pdfs = loader.load()
len(pdfs)


9

In [128]:
splitter = RecursiveCharacterTextSplitter(chunk_size= 100, chunk_overlap = 20)
splitted_data = splitter.split_documents(pdfs)
len(splitted_data)

266

In [129]:
embedding = OllamaEmbeddings(model="nomic-embed-text:latest")

In [130]:
vector_store = Chroma.from_documents(
    documents = splitted_data,
    embedding= embedding, 
)

In [131]:
query = "RAG"
data = vector_store.similarity_search(query=query)
data[0]

Document(id='47c695ab-6496-49f2-8a57-493bfded9a8f', metadata={'creationdate': '', 'producer': 'Skia/PDF m149 Google Docs Renderer', 'total_pages': 8, 'title': '10 RAG Architectures', 'page_label': '7', 'source': '../Data/10 RAG Architectures.pdf', 'creator': 'PyPDF', 'page': 6}, page_content='RAG)\n \nFocuses\n \non\n \nmaking\n \nRAG\n \noutputs\n \nexplainable\n.\n \n \nThe\n \nsystem:\n \n●\n \nshows')

In [132]:
context = ""
for doc in pdfs:
    context += doc.page_content + "\n"
    print(context)
    

Report Status    
Female
27 Years:
:
:
:
Age
Gender
Reported        
P
9/7/2025   4:56:00PM
DR NITIN NAHAR
474764803
Ms. NIKITA  CHUDHARY:
:
:
:
:
Name        
Lab No.    
Ref By 
Collected       
A/c Status 
10/7/2025  6:31:50PM
Final
Collected at            : Processed at             :BHOPAL CC-82
Mr Rachel V John Pata So Vitus John Mig 26 
Graund,Indrapuri, Phone: 8770817968
 
LPL - Bhopal Lab II
Plot No.05, Mandakini Housing Society, Near 
Apurti Shopping Mall, Kolar Main Road, 
Bhopal, M.P. -  462042
Test Report      
Test Name Results Units Bio. Ref. Interval
HEMOGRAM
(SLS Method, Sheath Flow DC Detection Method, Fluorescent Flow Cytometry, Calculated & Capillary Photometry)
Hemoglobin  12.00 - 15.00 g/dL13.10
Packed Cell Volume (PCV)  36.00 - 46.00 %39.50
RBC Count  3.80 - 4.80 mill/mm34.47
MCV  83.00 - 101.00 fL88.40
Mentzer Index   19.8
MCH  27.00 - 32.00 pg29.30
MCHC  31.50 - 34.50 g/dL33.20
Red Cell Distribution Width (RDW)  11.60 - 14.00 %11.90
Total Leukocyte Count (TLC)  

In [133]:
llm = ChatOllama(model="gemma4:31b-cloud")


In [134]:
# res = llm.invoke(f"""can you provide me the answer based on context for my question,
#                  Context : {context} and question:{query} """)

### Chain - context_generator | prompt | LLM | strparser

In [136]:
def get_context(query:str):
    data = vector_store.similarity_search(query=query)
    context = ""
    for doc in data:
      context += doc.page_content + "\n"
    
    
    return {
            "context": context,
             "question": query
    }
    
    

In [137]:
prompt = PromptTemplate.from_template("""
        You are a helpful assistant and provide answer based on the context and user question, and if you don't the answer, than you simply say that 'I don't know'                    
           Context:{context}
           Question:{question}
           
           """)



In [138]:
rag_chain = get_context | prompt | llm

In [147]:
res = rag_chain.invoke("what is RBC Count ?")


In [148]:
print(res.content)

The RBC Count is 4.47 mill/mm3.
